In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.text("table_name", "Customer")
dbutils.widgets.text("pipeline_run_id", "")

table_name = dbutils.widgets.get("table_name")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

In [0]:
from pyspark.sql.functions import *

metadata_table = f"{catalog_name}.{metadata_schema}.bronze_config"

metadata_df = (
    spark.table(metadata_table)
         .filter(col("active") == "Y")
)

In [0]:
display(metadata_df)

table_name,source_folder,target_table,gold_table,history_table,file_format,load_strategy,primary_key,source_system,watermark_column,compare_columns,scd_enabled,active
Customer,customer,customer,dim_customer,dim_customer_history,csv,INCREMENTAL,customer_id,Customer_CSV,registration_date,"first_name,last_name,email,phone,country,city,gender",Y,Y


In [0]:
config = (
    metadata_df
    .filter(
        upper(col("target_table")) == table_name.upper()
    )
    .first()
)

if config is None:
    raise Exception(f"No metadata found for {table_name}")

target_table = config["target_table"]
gold_table = config["gold_table"]
load_strategy = config["load_strategy"]

print("=" * 60)
print("Metadata Loaded")
print("=" * 60)
print("Table :", target_table)
print("Gold Table :", gold_table)
print("Load Strategy :", load_strategy)

# pipeline_name = "Generic_Gold_Load"
# pipeline_run_id = "pipeline_run_id"

Metadata Loaded
Table : customer
Gold Table : dim_customer
Load Strategy : INCREMENTAL


In [0]:
silver_table = (
    f"{catalog_name}."
    f"{silver_schema}."
    f"{target_table}"
)

df = spark.table(silver_table)

rows_read = df.count()

print("=" * 60)
print("Reading Silver Layer")
print("=" * 60)
print("Rows Read :", rows_read)

display(df.limit(10))

Reading Silver Layer
Rows Read : 903


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system
1001,Bruis,Myall,bmyall0@hibu.com,3604158081,United States,Vancouver,2022-09-23 11:10:45,2023-02-13 03:14:05,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,4d4a6739-9bdd-4754-9041-534fca1b7abf,7f1b0d87-beb8-4e10-b762-023fe277a227,Customer_CSV
1002,Ellene,MacSporran,emacsporran1@mapquest.com,9011721346,United States,Memphis,2022-10-15 18:01:12,2023-03-28 13:09:21,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,7ac2efdd-665f-4eec-8f4e-39f94d7b82a5,a2571a77-8746-493e-9d17-43ce1f27798b,Customer_CSV
1003,Isabelita,Powlesland,ipowlesland2@chicagotribune.com,7081890902,United Kingdom,Liverpool,2022-06-20 03:55:09,2022-09-08 06:25:57,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,72e707fb-35b8-4c57-9478-8c2000b01d7f,bdcb3d51-fa9b-411e-a4b7-a3967a787eeb,Customer_CSV
1004,Elden,Tonn,etonn3@t-online.de,8156364393,United States,Rockford,2023-03-08 09:56:41,2023-01-07 01:31:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2340d618-aa6b-40eb-8df3-bb2f899016ed,f66aa66b-fdcb-4cdb-bcf9-d902373e7893,Customer_CSV
1005,Edd,null,ediviny4@list-manage.com,4053842760,United States,null,2022-08-14 19:04:28,2022-04-23 15:11:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2efecdf8-b707-4d53-8291-bafb1226cc91,27df8248-b7a6-410b-b95b-7ae79521f65c,Customer_CSV
1006,Heinrik,Bulled,hbulled5@oaic.gov.au,8175639621,United States,Fort Worth,2022-11-26 05:31:07,2022-06-02 11:30:28,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,87c95270-188e-40b9-a84b-c093fb429b09,267f8eab-72b8-47e5-a22e-2bd4e0ee1d5c,Customer_CSV
1007,Greggory,Ellick,gellick6@bandcamp.com,8064064523,Australia,Sydney South,2022-05-10 08:45:30,2022-08-09 08:28:35,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,6b34b0aa-62a8-478d-b43d-d91841b570c3,4a9d12c8-90e0-4475-8a6b-37f857d44b4a,Customer_CSV
1008,Nolan,Dundridge,ndundridge7@sohu.com,2172361875,United States,Springfield,2023-04-14 07:37:43,2023-04-06 16:18:45,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,37804dcd-11af-4089-a312-bd5eeefb9858,5912fbb4-8f10-460a-bc65-74c6ebcc020e,Customer_CSV
1009,null,Standfield,tstandfield8@youtu.be,9459074722,null,Hatton,2022-11-15 20:47:58,2022-06-05 23:04:35,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,b02823e7-facb-4a8c-b98c-ba708bea9336,7a24b7bc-01ae-4fa0-aeeb-47c570e9c8f7,Customer_CSV
1010,Rozella,Hebdon,rhebdon9@sun.com,7129805160,United States,Omaha,2022-08-23 08:29:47,2023-02-13 08:06:10,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2edf7bf4-16ef-43e5-af80-38499d536851,58e98687-1e2f-4a46-95f3-d496a3be2b3e,Customer_CSV


In [0]:
gold_table = (
    f"{catalog_name}."
    f"{gold_schema}."
    f"{gold_table}"
)

print(gold_table)

dbw_insurance.insurance_gold.dim_customer


In [0]:
# display(metadata_df) 
# display(df) 
# print(gold_table)

In [0]:
# (
#     df.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable(gold_table)
# )

# write_mode = (
#     "overwrite"
#     if load_strategy.upper() == "FULL"
#     else "append"
# )

write_mode = "overwrite"
(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(gold_table)
)

rows_written = rows_read
gold_total_rows = spark.table(gold_table).count()

print("Rows Written :", rows_written)
print("Gold Total Rows :", gold_total_rows)

# rows_written = df.count()
# gold_total_rows = spark.table(gold_table).count()

Rows Written : 903
Gold Total Rows : 903


In [0]:
gold_df = spark.table(gold_table)

display(gold_df)

customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system
1001,Bruis,Myall,bmyall0@hibu.com,3604158081,United States,Vancouver,2022-09-23 11:10:45,2023-02-13 03:14:05,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,4d4a6739-9bdd-4754-9041-534fca1b7abf,7f1b0d87-beb8-4e10-b762-023fe277a227,Customer_CSV
1002,Ellene,MacSporran,emacsporran1@mapquest.com,9011721346,United States,Memphis,2022-10-15 18:01:12,2023-03-28 13:09:21,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,7ac2efdd-665f-4eec-8f4e-39f94d7b82a5,a2571a77-8746-493e-9d17-43ce1f27798b,Customer_CSV
1003,Isabelita,Powlesland,ipowlesland2@chicagotribune.com,7081890902,United Kingdom,Liverpool,2022-06-20 03:55:09,2022-09-08 06:25:57,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,72e707fb-35b8-4c57-9478-8c2000b01d7f,bdcb3d51-fa9b-411e-a4b7-a3967a787eeb,Customer_CSV
1004,Elden,Tonn,etonn3@t-online.de,8156364393,United States,Rockford,2023-03-08 09:56:41,2023-01-07 01:31:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2340d618-aa6b-40eb-8df3-bb2f899016ed,f66aa66b-fdcb-4cdb-bcf9-d902373e7893,Customer_CSV
1005,Edd,null,ediviny4@list-manage.com,4053842760,United States,null,2022-08-14 19:04:28,2022-04-23 15:11:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2efecdf8-b707-4d53-8291-bafb1226cc91,27df8248-b7a6-410b-b95b-7ae79521f65c,Customer_CSV
1006,Heinrik,Bulled,hbulled5@oaic.gov.au,8175639621,United States,Fort Worth,2022-11-26 05:31:07,2022-06-02 11:30:28,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,87c95270-188e-40b9-a84b-c093fb429b09,267f8eab-72b8-47e5-a22e-2bd4e0ee1d5c,Customer_CSV
1007,Greggory,Ellick,gellick6@bandcamp.com,8064064523,Australia,Sydney South,2022-05-10 08:45:30,2022-08-09 08:28:35,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,6b34b0aa-62a8-478d-b43d-d91841b570c3,4a9d12c8-90e0-4475-8a6b-37f857d44b4a,Customer_CSV
1008,Nolan,Dundridge,ndundridge7@sohu.com,2172361875,United States,Springfield,2023-04-14 07:37:43,2023-04-06 16:18:45,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,37804dcd-11af-4089-a312-bd5eeefb9858,5912fbb4-8f10-460a-bc65-74c6ebcc020e,Customer_CSV
1009,null,Standfield,tstandfield8@youtu.be,9459074722,null,Hatton,2022-11-15 20:47:58,2022-06-05 23:04:35,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,b02823e7-facb-4a8c-b98c-ba708bea9336,7a24b7bc-01ae-4fa0-aeeb-47c570e9c8f7,Customer_CSV
1010,Rozella,Hebdon,rhebdon9@sun.com,7129805160,United States,Omaha,2022-08-23 08:29:47,2023-02-13 08:06:10,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2edf7bf4-16ef-43e5-af80-38499d536851,58e98687-1e2f-4a46-95f3-d496a3be2b3e,Customer_CSV


In [0]:
silver_count = df.count()

gold_count = gold_df.count()

print("Silver Records :", silver_count)

print("Gold Records :", gold_count)

Silver Records : 903
Gold Records : 903


In [0]:
pipeline_name = "Generic_Gold_Load"

rows_read = silver_count
# rows_written = gold_count

write_audit(
    pipeline_name=pipeline_name,
    table_name=gold_table,
    load_type=load_strategy,
    rows_read=rows_read,
    rows_written=rows_written,
    status="SUCCESS",
    error_message="",
    pipeline_run_id=pipeline_run_id
)

In [0]:
display(

spark.table(

f"{catalog_name}.{audit_schema}.pipeline_audit"

)

.orderBy(col("start_time").desc())

)

pipeline_name,table_name,load_type,start_time,end_time,rows_read,rows_written,status,error_message,pipeline_run_id
Generic_Gold_Load,dbw_insurance.insurance_gold.dim_customer,INCREMENTAL,2026-07-17T11:11:15.661146Z,2026-07-17T11:11:15.661146Z,903,903,SUCCESS,,
Generic_Silver_Loader,customer,INCREMENTAL,2026-07-17T11:09:34.785938Z,2026-07-17T11:09:34.785938Z,2796,903,SUCCESS,,
Generic_Bronze_Loader,customer,INCREMENTAL,2026-07-17T11:07:39.381091Z,2026-07-17T11:07:39.381091Z,0,0,SUCCESS,,
Generic_Bronze_Loader,customer,INCREMENTAL,2026-07-17T11:07:00.761349Z,2026-07-17T11:07:00.761349Z,932,932,SUCCESS,,


In [0]:
print("=" * 70)
print("GENERIC GOLD LOADER COMPLETED")
print("=" * 70)

print(f"Table Name         : {target_table}")
print(f"Load Strategy      : {load_strategy}")
print(f"Rows Read          : {rows_read}")
print(f"Rows Written       : {rows_written}")
print(f"Gold Total Rows    : {gold_count}")
print(f"Pipeline Run ID    : {pipeline_run_id}")

print("=" * 70)

GENERIC GOLD LOADER COMPLETED
Table Name         : customer
Load Strategy      : INCREMENTAL
Rows Read          : 903
Rows Written       : 903
Gold Total Rows    : 903
Pipeline Run ID    : 


In [0]:
gold_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- pipeline_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)

